In [18]:
import redis
import time
import base64
import json
import csv
import random
import statistics

import numpy as np
import pandas as pd

In [19]:
def sliding_window(data, window_size, step):
    for start_row in range(0, len(data) - window_size + 1, step):
        yield data[start_row:start_row + window_size]        

In [20]:
#test = pd.read_csv("text_1.csv", header=None)
test = pd.read_csv("text_2.csv", header=None)
#test = pd.read_csv("text_3.csv", header=None)

In [21]:
test = test.transpose()
test[0]

0      b'{"ReceivedTopic":"","CorrelationID":"","Payl...
1      b'{"ReceivedTopic":"","CorrelationID":"","Payl...
2      b'{"ReceivedTopic":"","CorrelationID":"","Payl...
3      b'{"ReceivedTopic":"","CorrelationID":"","Payl...
4      b'{"ReceivedTopic":"","CorrelationID":"","Payl...
                             ...                        
995    b'{"ReceivedTopic":"","CorrelationID":"","Payl...
996    b'{"ReceivedTopic":"","CorrelationID":"","Payl...
997    b'{"ReceivedTopic":"","CorrelationID":"","Payl...
998    b'{"ReceivedTopic":"","CorrelationID":"","Payl...
999    b'{"ReceivedTopic":"","CorrelationID":"","Payl...
Name: 0, Length: 1000, dtype: object

In [22]:
humid = []
pm10 = []
pm25 = []
temp = []

for i in range(len(test)):
    val = test.values[i][0]
    val = val.replace("'", "")
    val = val.replace("b","",1)
    json_val = json.loads(val)
    
    res_payload = json_val['Payload']
    dec_res = base64.b64decode(res_payload)
    dec_res = dec_res.decode("UTF-8")
    str_test = dec_res.replace("'","\"")
    json_data = json.loads(str_test)
    
    humid.append(json_data['event']['readings'][0]['objectValue']['humidity'])
    pm10.append(json_data['event']['readings'][0]['objectValue']['pm10'])
    pm25.append(json_data['event']['readings'][0]['objectValue']['pm25'])
    temp.append(json_data['event']['readings'][0]['objectValue']['temperature'])

In [23]:
#절대값 차분
diff_humid = []
diff_pm10 = []
diff_pm25 = []
diff_temp = []

for i in range(len(humid)-1):
    diff_humid.append(abs(humid[i+1]-humid[i]))
    diff_pm10.append(abs(pm10[i+1]-pm10[i]))
    diff_pm25.append(abs(pm25[i+1]-pm25[i]))
    diff_temp.append(abs(temp[i+1]-temp[i]))

In [24]:
humid_series = pd.Series(diff_humid)

In [25]:
humid_series.describe()

count    999.000000
mean       1.595596
std        1.213080
min        0.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        4.000000
dtype: float64

In [31]:
humid_window_size = random.randint(1,1000)
humid_step = random.randint(1,1000)

pm10_window_size = random.randint(1,1000)
pm10_step = random.randint(1,1000)

pm25_window_size = random.randint(1,1000)
pm25_step = random.randint(1,1000)

temp_window_size = random.randint(1,1000)
temp_step = random.randint(1,1000)

humid_window = list(sliding_window(diff_humid,humid_window_size,humid_step))
pm10_window = list(sliding_window(diff_pm10,pm10_window_size,pm10_step))
pm25_window = list(sliding_window(diff_pm25,pm25_window_size,pm25_step))
temp_window = list(sliding_window(diff_temp,temp_window_size,temp_step))

avg_dif_humid = []
avg_dif_pm10 = []
avg_dif_pm25 = []
avg_dif_temp = []

for i in range(len(humid_window)):
    avg_dif_humid.append(statistics.mean(humid_window[i]))
for i in range(len(pm10_window)):
    avg_dif_pm10.append(statistics.mean(pm10_window[i]))
for i in range(len(pm25_window)):
    avg_dif_pm25.append(statistics.mean(pm25_window[i]))
for i in range(len(temp_window)):
    avg_dif_temp.append(statistics.mean(temp_window[i]))

total_avdf_humid = statistics.mean(avg_dif_humid)
total_avdf_pm10 = statistics.mean(avg_dif_pm10)
total_avdf_pm25 = statistics.mean(avg_dif_pm25)
total_avdf_temp = statistics.mean(avg_dif_temp)

total_av = (total_avdf_humid + total_avdf_pm10 + total_avdf_pm25 + total_avdf_temp) / 4

#변수별 차분평균 비교
cnt_val = []

for i in range(len(diff_temp)):
    if diff_humid[i] < total_avdf_humid and diff_pm10[i] < total_avdf_pm10 and diff_pm25[i] < total_avdf_pm25 and diff_temp[i] < total_avdf_temp:
        cnt_val.append(i)    
        
print("제거된 트래픽 수 : ", len(cnt_val))

제거된 트래픽 수 :  73


In [27]:
len(cnt_val)

73

In [28]:
total_avdf_humid = statistics.mean(avg_dif_humid)
total_avdf_pm10 = statistics.mean(avg_dif_pm10)
total_avdf_pm25 = statistics.mean(avg_dif_pm25)
total_avdf_temp = statistics.mean(avg_dif_temp)

total_av = (total_avdf_humid + total_avdf_pm10 + total_avdf_pm25 + total_avdf_temp) / 4

In [29]:
#변수별 차분평균의 평균과 비교
cnt_val = []

for i in range(len(diff_temp)):
    total_av_val = (diff_humid[i] +diff_pm10[i] + diff_pm25[i] + diff_temp[i]) / 4
    if  total_av_val <= total_av:
        cnt_val.append(i)
print("제거된 트래픽 수 : ", len(cnt_val))

제거된 트래픽 수 :  512
